In [ ]:
import sys
print(sys.version)

for pkg in ['geopandas', 'rasterio', 'skimage', 'numpy', 'pandas']:
    try:
        __import__(pkg)
        print(f"{pkg}: already available")
    except ImportError:
        print(f"{pkg}: NOT available, needs install")

In [ ]:
import sys
!{sys.executable} -m pip install --user geopandas rasterio scikit-image numpy pandas

In [ ]:
for pkg in ['geopandas', 'rasterio', 'skimage', 'numpy', 'pandas']:
    try:
        __import__(pkg)
        print(f"{pkg}: OK")
    except ImportError as e:
        print(f"{pkg}: FAILED - {e}")

In [ ]:
import sys
print("Python executable:", sys.executable)
print()
print("sys.path:")
for p in sys.path:
    print(" ", p)


In [ ]:
import sys
sys.path.insert(0, '__REDACTED_CLUSTER_HOME__/.local/lib/python3.12/site-packages')

for pkg in ['geopandas', 'rasterio', 'skimage', 'numpy', 'pandas']:
    try:
        __import__(pkg)
        print(f"{pkg}: OK")
    except ImportError as e:
        print(f"{pkg}: FAILED - {e}")

In [ ]:
import os
print(os.path.expanduser('~'))

In [ ]:
import os

# Where is this notebook actually running from?
print("Current working directory:", os.getcwd())
print()
print("Files here:")
for f in os.listdir():
    print(" ", f)

In [ ]:
import subprocess
result = subprocess.run(
    ["find", "__REDACTED_CLUSTER_PROJECT__", "-iname", "FINAL_grid.gpkg"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import geopandas as gpd

GRID_PATH = "__REDACTED_CLUSTER_PROJECT__/Imaging_data/FINAL_grid.gpkg"

grid = gpd.read_file(GRID_PATH)
print("Shape:", grid.shape)
print("Columns:", grid.columns.tolist())
grid.head()

In [ ]:
import subprocess
result = subprocess.run(
    ["find", "__REDACTED_CLUSTER_PROJECT__", "-iname", "Green_1_masked.tif"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import rasterio

TIFF_PATH = "__REDACTED_CLUSTER_PROJECT__/__REDACTED_WORKSPACE__/data_processed/Green_1_masked.tif"

with rasterio.open(TIFF_PATH) as src:
    print("Width x Height:", src.width, "x", src.height)
    print("Number of bands:", src.count)
    print("CRS:", src.crs)
    print("Bounds:", src.bounds)
    print("Data type:", src.dtypes)

In [ ]:
grid_reprojected = grid.to_crs("EPSG:4326")
print(grid_reprojected.crs)
print(grid_reprojected.head())

In [ ]:
from rasterio.mask import mask
import numpy as np

# pick plot 25 - the very first one in the grid, for a clean, traceable test
test_plot = grid_reprojected.iloc[0]
print("Testing Disease_Plot_ID:", test_plot['Disease_Plot_ID'])

with rasterio.open(TIFF_PATH) as src:
    out_image, out_transform = mask(src, [test_plot['geometry']], crop=True, nodata=np.nan, filled=True)
    pixels = out_image[0]

print("Extracted pixel array shape:", pixels.shape)
print("Number of pixels total:", pixels.size)
print("Number of non-NaN pixels:", np.sum(~np.isnan(pixels)))
print("Min/Max/Mean of valid pixels:", np.nanmin(pixels), np.nanmax(pixels), np.nanmean(pixels))

In [ ]:
import subprocess
result = subprocess.run(
    ["find", "__REDACTED_CLUSTER_PROJECT__", "-iname", "ALL_stats_1.csv"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import subprocess
result = subprocess.run(
    ["find", "__REDACTED_CLUSTER_HOME__", "-iname", "ALL_stats_1_JB_jack_data.csv"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
import pandas as pd
existing_stats = pd.read_csv("__REDACTED_CLUSTER_HOME__/ALL_stats_1_JB_jack_data.csv")
print(existing_stats.shape)
plot_25_check = existing_stats[existing_stats['Disease_Plot_ID'].astype(str) == '25']
green_cols = [c for c in plot_25_check.columns if 'Green' in c or 'green' in c]
print(green_cols)
print(plot_25_check[['Disease_Plot_ID'] + green_cols])

In [ ]:
print(existing_stats.columns.tolist())


In [ ]:
plot_25_ndvi = existing_stats[existing_stats['Disease_Plot_ID'].astype(str) == '25']['NDVI_1_mean']
print(plot_25_ndvi)

In [ ]:
NDVI_TIFF_PATH = "__REDACTED_CLUSTER_PROJECT__/__REDACTED_WORKSPACE__/data_processed/NDVI_1.tif"

with rasterio.open(NDVI_TIFF_PATH) as src:
    print("CRS:", src.crs)  # check this matches EPSG:4326 like before, or if it's back to 27700
    out_image, out_transform = mask(src, [test_plot['geometry']], crop=True, nodata=np.nan, filled=True)
    pixels_ndvi = out_image[0]

print("Non-NaN pixels:", np.sum(~np.isnan(pixels_ndvi)))
print("Mean NDVI (raw extraction):", np.nanmean(pixels_ndvi))

In [ ]:
with rasterio.open(NDVI_TIFF_PATH) as src:
    # all_touched=False (default) vs True changes which boundary pixels get included
    out_image_strict, _ = mask(src, [test_plot['geometry']], crop=True, nodata=np.nan, filled=True, all_touched=False)
    out_image_loose, _ = mask(src, [test_plot['geometry']], crop=True, nodata=np.nan, filled=True, all_touched=True)

print("all_touched=False: n=", np.sum(~np.isnan(out_image_strict[0])), " mean=", np.nanmean(out_image_strict[0]))
print("all_touched=True:  n=", np.sum(~np.isnan(out_image_loose[0])), " mean=", np.nanmean(out_image_loose[0]))

In [ ]:
# check if there's a difference between raw NDVI_1.tif values and what we'd expect from a masked-only extraction
print("Sample pixel values (first 20 non-nan):", pixels_ndvi[~np.isnan(pixels_ndvi)][:20])
print("Any suspiciously uniform low values (possible unmasked soil)?", np.sum(pixels_ndvi < 0.3))

In [ ]:
import subprocess

bands = ['Green', 'Red', 'RedEdge', 'NIR']
flights = [1, 2]

tiff_paths = {}

for flight in flights:
    for band in bands:
        fname = f"{band}_{flight}_masked.tif"
        result = subprocess.run(
            ["find", "__REDACTED_CLUSTER_PROJECT__/__REDACTED_WORKSPACE__/data_processed", "-iname", fname],
            capture_output=True, text=True
        )
        paths_found = result.stdout.strip().split('\n') if result.stdout.strip() else []
        tiff_paths[(band, flight)] = paths_found
        status = "OK" if len(paths_found) == 1 else f"PROBLEM ({len(paths_found)} matches)"
        print(f"{fname}: {status}")
        for p in paths_found:
            print("   ", p)

In [ ]:
import rasterio
from rasterio.mask import mask
import numpy as np
import pandas as pd
import time

BASE = "__REDACTED_CLUSTER_PROJECT__/__REDACTED_WORKSPACE__/data_processed"
bands = ['Green', 'Red', 'RedEdge', 'NIR']
flights = [1, 2]

# grid_reprojected should already exist in your session from last time
# if this is a fresh kernel, re-run the grid load + reproject cells first

results = []  # one dict per plot, collected across all bands/flights, merged at the end

# initialize with just the ID column
plot_ids = grid_reprojected['Disease_Plot_ID'].tolist()
geometries = grid_reprojected['geometry'].tolist()

output = pd.DataFrame({'Disease_Plot_ID': plot_ids})

for flight in flights:
    for band in bands:
        tiff_path = f"{BASE}/{band}_{flight}_masked.tif"
        print(f"\nProcessing {band} flight {flight}...")
        t0 = time.time()

        col_prefix = f"{band}_{flight}"
        stats = {
            f"{col_prefix}_p10": [],
            f"{col_prefix}_p90": [],
            f"{col_prefix}_std": [],
            f"{col_prefix}_min": [],
            f"{col_prefix}_max": [],
        }

        with rasterio.open(tiff_path) as src:
            for geom in geometries:
                try:
                    out_image, _ = mask(src, [geom], crop=True, nodata=np.nan, filled=True)
                    pixels = out_image[0]
                    valid = pixels[~np.isnan(pixels)]

                    if valid.size == 0:
                        stats[f"{col_prefix}_p10"].append(np.nan)
                        stats[f"{col_prefix}_p90"].append(np.nan)
                        stats[f"{col_prefix}_std"].append(np.nan)
                        stats[f"{col_prefix}_min"].append(np.nan)
                        stats[f"{col_prefix}_max"].append(np.nan)
                    else:
                        stats[f"{col_prefix}_p10"].append(np.percentile(valid, 10))
                        stats[f"{col_prefix}_p90"].append(np.percentile(valid, 90))
                        stats[f"{col_prefix}_std"].append(np.std(valid))
                        stats[f"{col_prefix}_min"].append(np.min(valid))
                        stats[f"{col_prefix}_max"].append(np.max(valid))
                except Exception as e:
                    # plot geometry might not overlap this raster at all (e.g. flight coverage differs) - log and continue
                    print(f"  Failed on a plot: {e}")
                    stats[f"{col_prefix}_p10"].append(np.nan)
                    stats[f"{col_prefix}_p90"].append(np.nan)
                    stats[f"{col_prefix}_std"].append(np.nan)
                    stats[f"{col_prefix}_min"].append(np.nan)
                    stats[f"{col_prefix}_max"].append(np.nan)

        for col, vals in stats.items():
            output[col] = vals

        elapsed = time.time() - t0
        print(f"  Done in {elapsed:.1f}s")

print("\nFinal shape:", output.shape)
output.head()

In [ ]:
bands = ['Green', 'Red', 'RedEdge', 'NIR']
flights = [1, 2]

# CV = coefficient of variation, std relative to mean — scale-invariant,
# defensible number for "how patchy is this plot" regardless of flight's absolute scale
for flight in flights:
    for band in bands:
        prefix = f"{band}_{flight}"
        mean_col = None
        # we didn't extract mean directly - reconstruct as midpoint isn't right,
        # so instead let's derive CV from p90-p10 spread relative to the plot's own scale
        output[f"{prefix}_cv_range"] = (output[f"{prefix}_p90"] - output[f"{prefix}_p10"]) / (
            (output[f"{prefix}_p90"] + output[f"{prefix}_p10"]) / 2
        )

print("New columns added:", [c for c in output.columns if 'cv_range' in c])
output[[c for c in output.columns if 'cv_range' in c]].describe()

In [ ]:
import subprocess
result = subprocess.run(
    ["find", "__REDACTED_CLUSTER_PROJECT__", "-iname", "final_disease_data.csv"],
    capture_output=True, text=True
)
print(result.stdout)
print(result.stderr)

In [ ]:
# quick check: does CV separate healthy/moderate/diseased sensibly, WITHIN each flight?
# using final_disease_data or equivalent - assumes you have plot-level disease scores loaded
# adjust column names if yours differ# find the bad values in output's Disease_Plot_ID
bad_mask = ~output['Disease_Plot_ID'].astype(str).str.match(r'^\d+$')
print("Number of bad rows:", bad_mask.sum())
print(output.loc[bad_mask, 'Disease_Plot_ID'])

disease_scores = pd.read_csv("__REDACTED_CLUSTER_PROJECT__/Imaging_data/final_disease_data.csv")
print(disease_scores.shape)
print(disease_scores.columns.tolist())
disease_scores.head()

In [ ]:
# find the bad values in output's Disease_Plot_ID
bad_mask = ~output['Disease_Plot_ID'].astype(str).str.match(r'^\d+$')
print("Number of bad rows:", bad_mask.sum())
print(output.loc[bad_mask, 'Disease_Plot_ID'])

In [ ]:
# also check the source column in the grid itself
bad_mask_grid = ~grid_reprojected['Disease_Plot_ID'].astype(str).str.match(r'^\d+$')
print("Bad rows in grid_reprojected:", bad_mask_grid.sum())
print(grid_reprojected.loc[bad_mask_grid, ['id', 'QGIS_ID', 'Disease_Plot_ID']])

In [ ]:
output_clean = output[output['Disease_Plot_ID'].astype(str).str.match(r'^\d+$')].copy()
output_clean['Disease_Plot_ID'] = output_clean['Disease_Plot_ID'].astype(int)

print("Original rows:", len(output), "-> After dropping guard plots:", len(output_clean))
# drop the CV columns - didn't pass the sanity check, direction flips between flights
cv_cols_to_drop = [c for c in output_clean.columns if 'cv_range' in c]
print("Dropping:", cv_cols_to_drop)

output_final = output_clean.drop(columns=cv_cols_to_drop)

print("\nFinal shape:", output_final.shape)
print("Columns:", output_final.columns.tolist())

# sanity check: no unexpected NaNs beyond what's expected from failed masks, if any
print("\nNaN counts per column:")
print(output_final.isna().sum()[output_final.isna().sum() > 0])
disease_scores['Disease_Plot_ID'] = disease_scores['Disease_Plot_ID'].astype(int)

for flight in [1, 2]:
    flight_label = f"Flight {flight}"
    flight_scores = disease_scores[disease_scores['Flight'] == flight_label][['Disease_Plot_ID', 'severity']]

    merged = output_clean.merge(flight_scores, on='Disease_Plot_ID', how='inner')

    print(f"\n=== Flight {flight} — CV by severity class ===")
    print(f"n = {len(merged)}")

    cv_cols = [c for c in merged.columns if f"_{flight}_cv_range" in c]
    print(merged.groupby('severity')[cv_cols].mean())

In [ ]:
# drop the CV columns - didn't pass the sanity check, direction flips between flights
cv_cols_to_drop = [c for c in output_clean.columns if 'cv_range' in c]
print("Dropping:", cv_cols_to_drop)

output_final = output_clean.drop(columns=cv_cols_to_drop)

print("\nFinal shape:", output_final.shape)
print("Columns:", output_final.columns.tolist())

# sanity check: no unexpected NaNs beyond what's expected from failed masks, if any
print("\nNaN counts per column:")
print(output_final.isna().sum()[output_final.isna().sum() > 0])

In [ ]:
output_path = "__REDACTED_CLUSTER_HOME__/texture_features_percentiles.csv"
output_final.to_csv(output_path, index=False)
print(f"Saved to {output_path}")
print(f"Shape: {output_final.shape}")

In [ ]:
from skimage.feature import graycomatrix, graycoprops
import numpy as np
import rasterio
from rasterio.mask import mask

BASE = "__REDACTED_CLUSTER_PROJECT__/__REDACTED_WORKSPACE__/data_processed"
GLCM_LEVELS = 16

def extract_glcm_features(pixels_2d, levels=GLCM_LEVELS):
    """
    pixels_2d: 2D array with NaN for masked-out (soil) pixels
    Returns dict of GLCM texture stats, or None if too few valid pixels
    """
    valid_mask = ~np.isnan(pixels_2d)
    n_valid = valid_mask.sum()

    if n_valid < 50:  # arbitrary but reasonable floor - GLCM on tiny plots is unreliable
        return None

    # bin into discrete gray levels using only the valid pixel range
    valid_vals = pixels_2d[valid_mask]
    vmin, vmax = valid_vals.min(), valid_vals.max()

    if vmax == vmin:  # degenerate: uniform plot, no texture possible
        return None

    # scale valid pixels to 0..levels-1, keep NaN pixels as a level skimage will ignore via masking
    binned = np.zeros_like(pixels_2d, dtype=np.uint8)
    binned[valid_mask] = np.clip(
        ((valid_vals - vmin) / (vmax - vmin) * (levels - 1)).astype(np.uint8),
        0, levels - 1
    )
    # soil/NaN pixels: set to 0 - NOTE this is a simplification, addressed in scale-up discussion below
    binned[~valid_mask] = 0

    # true 2D GLCM: distance=1 pixel, averaged across 4 directions (0°, 45°, 90°, 135°)
    # this is the fix for yesterday's bug - real 2D adjacency, not flattened 1D
    glcm = graycomatrix(
        binned, distances=[1], angles=[0, np.pi/4, np.pi/2, 3*np.pi/4],
        levels=levels, symmetric=True, normed=True
    )

    return {
        'glcm_contrast': graycoprops(glcm, 'contrast').mean(),
        'glcm_homogeneity': graycoprops(glcm, 'homogeneity').mean(),
        'glcm_energy': graycoprops(glcm, 'energy').mean(),
        'glcm_correlation': graycoprops(glcm, 'correlation').mean(),
    }

# test on plot 25, Green band, flight 1 - same plot we already validated for percentiles
TIFF_PATH = f"{BASE}/Green_1_masked.tif"
test_plot = grid_reprojected.iloc[0]

with rasterio.open(TIFF_PATH) as src:
    out_image, _ = mask(src, [test_plot['geometry']], crop=True, nodata=np.nan, filled=True)
    pixels = out_image[0]

print("Plot shape:", pixels.shape, "| valid pixels:", np.sum(~np.isnan(pixels)))

result = extract_glcm_features(pixels)
print("\nGLCM features for plot 25, Green, Flight 1:")
for k, v in result.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
def get_valid_bbox(pixels_2d):
    """Crop to the tightest bounding box containing valid (non-NaN) pixels"""
    valid_mask = ~np.isnan(pixels_2d)
    rows = np.any(valid_mask, axis=1)
    cols = np.any(valid_mask, axis=0)
    rmin, rmax = np.where(rows)[0][[0, -1]]
    cmin, cmax = np.where(cols)[0][[0, -1]]
    return pixels_2d[rmin:rmax+1, cmin:cmax+1]

pixels_cropped = get_valid_bbox(pixels)
print("Cropped shape:", pixels_cropped.shape, "vs original:", pixels.shape)
print("Valid pixels in crop:", np.sum(~np.isnan(pixels_cropped)), "/ crop total:", pixels_cropped.size)

result_cropped = extract_glcm_features(pixels_cropped)
print("\nGLCM features, TIGHTLY CROPPED, plot 25, Green, Flight 1:")
for k, v in result_cropped.items():
    print(f"  {k}: {v:.4f}")

print("\nOriginal (full array, more soil-boundary padding):")
for k, v in result.items():
    print(f"  {k}: {v:.4f}")

In [ ]:
# validate crop-vs-full stability across a handful of plots with different likely mask shapes
import pandas as pd

sample_ids = [25, 50, 75, 100, 125]  # first 5 plots, arbitrary but varied
sample_check = []

with rasterio.open(f"{BASE}/Green_1_masked.tif") as src:
    for pid in sample_ids:
        plot_row = grid_reprojected[grid_reprojected['Disease_Plot_ID'].astype(str) == str(pid)]
        if len(plot_row) == 0:
            continue
        geom = plot_row.iloc[0]['geometry']

        out_image, _ = mask(src, [geom], crop=True, nodata=np.nan, filled=True)
        px = out_image[0]

        full_result = extract_glcm_features(px)
        cropped_result = extract_glcm_features(get_valid_bbox(px))

        if full_result is None or cropped_result is None:
            print(f"Plot {pid}: skipped (degenerate)")
            continue

        row = {'plot': pid, 'n_valid': np.sum(~np.isnan(px))}
        for k in full_result:
            row[f"{k}_full"] = full_result[k]
            row[f"{k}_crop"] = cropped_result[k]
            row[f"{k}_diff"] = abs(full_result[k] - cropped_result[k])
        sample_check.append(row)

check_df = pd.DataFrame(sample_check)
diff_cols = [c for c in check_df.columns if c.endswith('_diff')]
print(check_df[['plot', 'n_valid'] + diff_cols])
print("\nMax diff across all:", check_df[diff_cols].max().max())

In [ ]:
def extract_glcm_features_masked(pixels_2d, levels=GLCM_LEVELS, distance=1):
    """
    NaN-aware GLCM: only counts co-occurrence pairs where BOTH pixels are valid.
    Computes manually rather than via skimage.graycomatrix (which has no NaN support),
    but uses the same 4-direction averaging (0°, 45°, 90°, 135°) for consistency.
    """
    valid_mask = ~np.isnan(pixels_2d)
    n_valid = valid_mask.sum()

    if n_valid < 50:
        return None

    valid_vals = pixels_2d[valid_mask]
    vmin, vmax = valid_vals.min(), valid_vals.max()
    if vmax == vmin:
        return None

    # bin to gray levels; masked pixels get level -1 (sentinel, excluded from pairs)
    binned = np.full(pixels_2d.shape, -1, dtype=np.int16)
    binned[valid_mask] = np.clip(
        ((valid_vals - vmin) / (vmax - vmin) * (levels - 1)).astype(np.int16),
        0, levels - 1
    )

    offsets = {
        '0':   (0, 1),
        '45':  (-1, 1),
        '90':  (-1, 0),
        '135': (-1, -1),
    }

    glcm_sum = np.zeros((levels, levels), dtype=np.float64)
    pair_count = 0

    for dr, dc in offsets.values():
        rows, cols = binned.shape
        r0_start, r0_end = max(0, -dr), min(rows, rows - dr)
        c0_start, c0_end = max(0, -dc), min(cols, cols - dc)

        a = binned[r0_start:r0_end, c0_start:c0_end]
        b = binned[r0_start+dr:r0_end+dr, c0_start+dc:c0_end+dc]

        valid_pair = (a >= 0) & (b >= 0)
        a_valid = a[valid_pair]
        b_valid = b[valid_pair]

        if a_valid.size == 0:
            continue

        # accumulate symmetric co-occurrence counts
        idx = a_valid.astype(np.int64) * levels + b_valid.astype(np.int64)
        counts = np.bincount(idx, minlength=levels*levels).reshape(levels, levels)
        glcm_sum += counts + counts.T  # symmetric
        pair_count += a_valid.size * 2

    if pair_count == 0:
        return None

    glcm_norm = glcm_sum / glcm_sum.sum()

    # compute props manually from the normalized co-occurrence matrix
    i_idx, j_idx = np.meshgrid(np.arange(levels), np.arange(levels), indexing='ij')
    contrast = np.sum(glcm_norm * (i_idx - j_idx) ** 2)
    homogeneity = np.sum(glcm_norm / (1.0 + (i_idx - j_idx) ** 2))
    energy = np.sqrt(np.sum(glcm_norm ** 2))

    mu_i = np.sum(i_idx * glcm_norm)
    mu_j = np.sum(j_idx * glcm_norm)
    sigma_i = np.sqrt(np.sum(((i_idx - mu_i) ** 2) * glcm_norm))
    sigma_j = np.sqrt(np.sum(((j_idx - mu_j) ** 2) * glcm_norm))
    if sigma_i > 0 and sigma_j > 0:
        correlation = np.sum(glcm_norm * (i_idx - mu_i) * (j_idx - mu_j)) / (sigma_i * sigma_j)
    else:
        correlation = 0.0

    return {
        'glcm_contrast': contrast,
        'glcm_homogeneity': homogeneity,
        'glcm_energy': energy,
        'glcm_correlation': correlation,
    }

# re-validate on the same 5 plots against the crop-only version, to confirm this actually fixes the drift
sample_check_v2 = []
with rasterio.open(f"{BASE}/Green_1_masked.tif") as src:
    for pid in sample_ids:
        plot_row = grid_reprojected[grid_reprojected['Disease_Plot_ID'].astype(str) == str(pid)]
        geom = plot_row.iloc[0]['geometry']
        out_image, _ = mask(src, [geom], crop=True, nodata=np.nan, filled=True)
        px = out_image[0]

        masked_result = extract_glcm_features_masked(px)
        if masked_result is None:
            print(f"Plot {pid}: degenerate, skipped")
            continue
        row = {'plot': pid}
        row.update(masked_result)
        sample_check_v2.append(row)

check_df_v2 = pd.DataFrame(sample_check_v2)
print(check_df_v2)

In [ ]:
import time
t0 = time.time()
extract_glcm_features_masked(pixels)  # reuse plot 25's array already in memory
elapsed = time.time() - t0
print(f"Time per plot: {elapsed:.3f}s")
print(f"Estimated total for 4800 extractions: {elapsed * 4800 / 60:.1f} minutes")

In [ ]:
import time

bands = ['Green', 'Red', 'RedEdge', 'NIR']
flights = [1, 2]

plot_ids = grid_reprojected['Disease_Plot_ID'].tolist()
geometries = grid_reprojected['geometry'].tolist()

glcm_output = pd.DataFrame({'Disease_Plot_ID': plot_ids})

for flight in flights:
    for band in bands:
        tiff_path = f"{BASE}/{band}_{flight}_masked.tif"
        print(f"\nProcessing GLCM for {band} flight {flight}...")
        t0 = time.time()

        col_prefix = f"{band}_{flight}"
        stats = {
            f"{col_prefix}_glcm_contrast": [],
            f"{col_prefix}_glcm_homogeneity": [],
            f"{col_prefix}_glcm_energy": [],
            f"{col_prefix}_glcm_correlation": [],
        }

        with rasterio.open(tiff_path) as src:
            for geom in geometries:
                try:
                    out_image, _ = mask(src, [geom], crop=True, nodata=np.nan, filled=True)
                    pixels = out_image[0]
                    result = extract_glcm_features_masked(pixels)

                    if result is None:
                        for k in stats:
                            stats[k].append(np.nan)
                    else:
                        stats[f"{col_prefix}_glcm_contrast"].append(result['glcm_contrast'])
                        stats[f"{col_prefix}_glcm_homogeneity"].append(result['glcm_homogeneity'])
                        stats[f"{col_prefix}_glcm_energy"].append(result['glcm_energy'])
                        stats[f"{col_prefix}_glcm_correlation"].append(result['glcm_correlation'])
                except Exception as e:
                    print(f"  Failed on a plot: {e}")
                    for k in stats:
                        stats[k].append(np.nan)

        for col, vals in stats.items():
            glcm_output[col] = vals

        elapsed = time.time() - t0
        n_nan = pd.isna(stats[f"{col_prefix}_glcm_contrast"]).sum()
        print(f"  Done in {elapsed:.1f}s | {n_nan} degenerate/failed plots")

print("\nFinal GLCM shape:", glcm_output.shape)
glcm_output.head()

In [ ]:
# 1. filter guard plots (same pattern as percentile extraction)
glcm_clean = glcm_output[glcm_output['Disease_Plot_ID'].astype(str).str.match(r'^\d+$')].copy()
glcm_clean['Disease_Plot_ID'] = glcm_clean['Disease_Plot_ID'].astype(int)

print("Original rows:", len(glcm_output), "-> After dropping guard plots:", len(glcm_clean))

# 2. check for NaNs (degenerate plots) before export
nan_summary = glcm_clean.isna().sum()
print("\nColumns with NaNs (degenerate plots):")
print(nan_summary[nan_summary > 0])

# 3. sanity check: does GLCM separate healthy/moderate/diseased sensibly, same test as CV before
for flight in [1, 2]:
    flight_label = f"Flight {flight}"
    flight_scores = disease_scores[disease_scores['Flight'] == flight_label][['Disease_Plot_ID', 'severity']]
    merged = glcm_clean.merge(flight_scores, on='Disease_Plot_ID', how='inner')

    print(f"\n=== Flight {flight} — GLCM by severity class (n={len(merged)}) ===")
    glcm_cols = [c for c in merged.columns if f"_{flight}_glcm_" in c]
    print(merged.groupby('severity')[glcm_cols].mean())

In [ ]:
percentile_cols_1 = [c for c in output_final.columns if '_1_' in c]
percentile_cols_2 = [c for c in output_final.columns if '_2_' in c]

for flight in [1, 2]:
    flight_label = f"Flight {flight}"
    flight_scores = disease_scores[disease_scores['Flight'] == flight_label][['Disease_Plot_ID', 'severity']]

    merged = output_final.merge(flight_scores, on='Disease_Plot_ID', how='inner')

    print(f"\n=== Flight {flight} — raw percentile stats by severity (n={len(merged)}) ===")
    cols = [c for c in merged.columns if f"_{flight}_" in c]
    print(merged.groupby('severity')[cols].mean().T)

In [ ]:
selection_results = []

for col_base in ['Green', 'Red', 'RedEdge', 'NIR']:
    for stat in ['p10', 'p90', 'std', 'min', 'max']:
        col1 = f"{col_base}_1_{stat}"
        col2 = f"{col_base}_2_{stat}"

        # get flight 1 group means
        f1_scores = disease_scores[disease_scores['Flight'] == 'Flight 1'][['Disease_Plot_ID', 'severity']]
        m1 = output_final.merge(f1_scores, on='Disease_Plot_ID', how='inner')
        g1 = m1.groupby('severity')[col1].mean()

        f2_scores = disease_scores[disease_scores['Flight'] == 'Flight 2'][['Disease_Plot_ID', 'severity']]
        m2 = output_final.merge(f2_scores, on='Disease_Plot_ID', how='inner')
        g2 = m2.groupby('severity')[col2].mean()

        diff1 = g1['diseased'] - g1['healthy']
        diff2 = g2['diseased'] - g2['healthy']

        same_direction = (diff1 > 0) == (diff2 > 0)
        # normalize each diff by that flight's overall std for the column, so we can compare magnitude fairly
        norm_diff1 = diff1 / output_final[col1].std()
        norm_diff2 = diff2 / output_final[col2].std()

        avg_effect = (abs(norm_diff1) + abs(norm_diff2)) / 2

        selection_results.append({
            'band': col_base,
            'stat': stat,
            'diff_flight1': diff1,
            'diff_flight2': diff2,
            'same_direction': same_direction,
            'avg_normalized_effect': avg_effect
        })

selection_df = pd.DataFrame(selection_results).sort_values(
    ['same_direction', 'avg_normalized_effect'], ascending=[False, False]
)
pd.set_option('display.max_rows', 50)
print(selection_df.to_string(index=False))

In [ ]:
selected_cols = ['Disease_Plot_ID', 'NIR_1_max', 'NIR_2_max', 'RedEdge_1_p90', 'RedEdge_2_p90', 'Red_1_min', 'Red_2_min']
texture_selected = output_final[selected_cols]
texture_selected.to_csv('__REDACTED_CLUSTER_HOME__/texture_features_selected.csv', index=False)
print("Saved. Shape:", texture_selected.shape)
texture_selected.head()


In [ ]:
import sys
sys.path.insert(0, '__REDACTED_CLUSTER_HOME__/.local/lib/python3.12/site-packages')
import rasterio, geopandas
print("rasterio:", rasterio.__version__)
print("geopandas:", geopandas.__version__)